In [1]:
import math
import numpy as np
import torch

from sklearn.decomposition import PCA

from datasets.berlinurbangradient import BerlinUrbanGradient
from metrics import psnr, sa

In [2]:
dataset_path = "./datasets/berlin-urban-gradient/"

In [3]:
test_dataset = BerlinUrbanGradient(dataset_path, split="test", transform=None)
src_channels = 111

psnr_metric = psnr.PeakSignalToNoiseRatio()
sa_metric = sa.SpectralAngle()

In [4]:
for cratio in [4, 8, 16, 27.75, 55.5, 111]:
    cr_list = []
    psnr_list = []
    sa_list = []

    latent_channels = int(math.ceil(src_channels/cratio))
    pca = PCA(n_components=latent_channels)

    for data_org in test_dataset:
        # data_in = 10_000 * data_org
        # data_in = torch.Tensor.numpy(data_in)
        # data_in = data_in.astype(np.uint16)

        data_in = data_org
        data_in = data_in.flatten(1, 2)
        data_in = data_in.T
        data_in = data_in.numpy()

        data_lat = pca.fit_transform(data_in)

        data_out = pca.inverse_transform(data_lat)
        data_out = data_out.T
        data_out = np.resize(data_out, (111, 80, 80))

        data_rec = torch.from_numpy(data_out)

        size_raw = data_in.nbytes # / 1024 / 1024
        # print(f"raw size:\t{size_raw} MB")

        size_lat = data_lat.nbytes # / 1024 / 1024
        # print(f"size_lat size:\t{size_lat} MB")

        org_size = src_channels * 80 * 80
        lat_size = latent_channels * 80 * 80 + src_channels * latent_channels + src_channels # pca coefficients + pca basis + mean vector
        cr = org_size / lat_size 
        # print(f"CR:\t\t{cr}")

        psnr = psnr_metric(data_org, data_rec)
        # print(f"PSNR:\t\t{psnr} dB")

        sa = sa_metric(data_org, data_rec)
        # print(f"SA:\t\t{sa} °")

        cr_list.append(cr)
        psnr_list.append(psnr)
        sa_list.append(sa)

    cr_avg = np.mean(cr_list)
    psnr_avg = np.mean(psnr_list)
    sa_avg = np.nanmean(sa_list)

    print(latent_channels)
    print(f"cr_avg:\t\t{cr_avg}")
    print(f"psnr_avg:\t{psnr_avg} dB")
    print(f"sa_avg: \t{sa_avg} °")
    print("===")

28
cr_avg:		3.894331182607075
psnr_avg:	64.44233703613281 dB
sa_avg: 	0.23478899896144867 °
===
14
cr_avg:		7.783925929984112
psnr_avg:	58.13358688354492 dB
sa_avg: 	0.4776691496372223 °
===
7
cr_avg:		15.5489406408685
psnr_avg:	51.80513000488281 dB
sa_avg: 	0.9450373649597168 °
===
4
cr_avg:		27.161154654941694
psnr_avg:	46.162532806396484 dB
sa_avg: 	1.8503916263580322 °
===
2
cr_avg:		54.09274347064646
psnr_avg:	37.72783279418945 dB
sa_avg: 	4.697499752044678 °
===
1
cr_avg:		107.27876774388402
psnr_avg:	28.04143524169922 dB
sa_avg: 	14.32550048828125 °
===
